[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C63_ML_System_Design_Course/01_requirements/01_requirements_metrics.ipynb)

# 01 · 需求澄清与指标定义（澄清清单 / 三层指标一致性 / 代价矩阵工作点 / 延迟预算分解）

目标：把「需求澄清」从一句寒暄，变成**几个可以运行、可以断言的小工具**。

本 notebook 你会亲手实现：
1. **环境自检**
2. **需求澄清问题清单生成器** —— 七个维度的问题模板 + 覆盖度检查
3. **三层指标映射与一致性检查** —— 用同一份逐类数据，分别按评测集分布/真实分布/代价加权算三遍
4. **代价矩阵下的最优工作点**（部分留作练习）
5. **延迟预算分解器** —— 把端到端预算拆到各子系统，支持每个子系统的最低保证（floor）
6. **四道练习**：一致性检查 / 最优阈值网格搜索 / 范围裁剪报告 / 非功能需求覆盖度检查

> 心智模型：**模糊需求 -> 可验收规格；模型指标涨了不等于业务指标涨了；阈值是需求澄清的产出，不是训练完才想起来的事。**

## 0 · 环境自检

In [ ]:
import sys
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)
assert sys.version_info >= (3, 8), '需要 Python 3.8+'
print('\n✅ 环境自检通过：本课不需要 GPU、不需要联网。')

## 1 · 需求澄清问题清单生成器

把 §2 的七个维度做成数据结构；`coverage_check(answers)` 检查一次澄清对话的记录里
**哪些维度还没有留下一句非空的答案**——这就是「可验收规格」的可操作定义。

In [ ]:
CHECKLIST = [
    ('who',         '① 用户是谁',     '最终使用者/下游消费方是谁？是终端用户还是内部系统？'),
    ('scenario',    '② 什么场景',     '在什么条件下运行？白天/夜晚？城市/高速？哪个地区的标志体系？'),
    ('success',     '③ 什么算成功',   '用什么指标衡量？达到多少算达标？'),
    ('failure',     '④ 什么算失败',   '错误具体长什么样？漏检/误检/错分/定位不准/太慢？'),
    ('cost',        '⑤ 失败的代价',   '漏检一次和误检一次，代价量级差多少？'),
    ('constraints', '⑥ 有什么约束',   '延迟预算/算力/数据规模/标注预算/团队规模/时间线？'),
    ('non_goals',   '⑦ 什么不做',     '第一版明确不做什么？'),
]

def generate_questions():
    """返回七个维度的 (key, 维度名, 问题模板)。"""
    return list(CHECKLIST)

def coverage_check(answers):
    """answers: {key: 回答文本}。返回还缺答案（空或未提供）的维度 key 列表，保持 CHECKLIST 顺序。"""
    return [k for k, _, _ in CHECKLIST if not answers.get(k, '').strip()]

for k, name, q in generate_questions():
    print(f'{name:<10} {q}')

# —— 一次不完整的澄清记录 ——
answers_partial = {
    'who': '规控模块与 VLA 感知接口（见 C59-03）',
    'scenario': '城市 + 高速，覆盖 GB 5768',
    'success': '',                       # 还没定义
    'failure': '漏检 / 误检 / 错分',
    'cost': '',                          # 还没定义
    'constraints': '端到端延迟预算 20ms',
    'non_goals': '第一版不做电子可变限速牌',
}
missing = coverage_check(answers_partial)
assert missing == ['success', 'cost']
print(f'\n覆盖度检查：还缺 {missing} 两项 —— 这两项恰好是 §3-§5 要展开的内容。')
print('✅ 清单生成器就位：把一次真实的模拟澄清对话喂进来，就能自动指出漏了哪一格。')

## 2 · 三层指标映射与一致性检查（数值例子）

复现正文 §4 的例子：六类标志的逐类召回率固定，但**评测集分布**、**真实路测分布**、**按业务代价加权**
三种权重会给出不同的汇总结论。

In [ ]:
# 逐类数据：recall 是模型指标（与分布无关）；eval_share/real_share 是两种流量分布；cost 是漏检的相对业务代价
TSR_CLASSES = {
    'speed_limit':        {'recall': 0.97, 'eval_share': 0.40, 'real_share': 0.55, 'cost': 1},
    'stop_sign':          {'recall': 0.95, 'eval_share': 0.25, 'real_share': 0.20, 'cost': 1},
    'yield_sign':         {'recall': 0.93, 'eval_share': 0.15, 'real_share': 0.12, 'cost': 1},
    'construction_temp':  {'recall': 0.65, 'eval_share': 0.05, 'real_share': 0.03, 'cost': 8},
    'electronic_sign':    {'recall': 0.55, 'eval_share': 0.02, 'real_share': 0.05, 'cost': 10},
    'other_info':         {'recall': 0.90, 'eval_share': 0.13, 'real_share': 0.05, 'cost': 1},
}
assert abs(sum(v['eval_share'] for v in TSR_CLASSES.values()) - 1.0) < 1e-9
assert abs(sum(v['real_share'] for v in TSR_CLASSES.values()) - 1.0) < 1e-9

def weighted_recall(data, share_key):
    """模型指标：按给定分布加权的召回率。"""
    return sum(v[share_key] * v['recall'] for v in data.values())

def business_cost(data, share_key):
    """业务指标的代理：按 分布 x 漏检率 x 代价 加权求和，越低越好。"""
    return sum(v[share_key] * (1 - v['recall']) * v['cost'] for v in data.values())

wr_eval = weighted_recall(TSR_CLASSES, 'eval_share')
wr_real = weighted_recall(TSR_CLASSES, 'real_share')
bc_eval = business_cost(TSR_CLASSES, 'eval_share')
bc_real = business_cost(TSR_CLASSES, 'real_share')

assert np.isclose(wr_eval, 0.9255)
assert np.isclose(wr_real, 0.9271)
assert np.isclose(bc_eval, 0.278)
assert np.isclose(bc_real, 0.3489)

print(f'模型指标（加权召回率）  评测集分布 {wr_eval:.4f}  ->  真实分布 {wr_real:.4f}   变化 {(wr_real-wr_eval)*100:+.2f} pp')
print(f'业务指标（代价加权）    评测集分布 {bc_eval:.4f}  ->  真实分布 {bc_real:.4f}   相对变化 {(bc_real-bc_eval)/bc_eval*100:+.1f}%')
print('\n✅ 模型指标几乎不变，业务代价却上升了 25%+ —— 这就是 §4 说的"指标背离"，不是抽象说法。')

## 3 · 代价矩阵与工作点（网格搜索的构件）

先给出 `cost_at(t, cfp, cfn)` 这个基础构件（给定阈值 `t` 计算期望代价），
**最优阈值的搜索留给练习 2**。

In [ ]:
# 合成的正/负样本置信度分数（正 = 有标志，负 = 无标志/其他类）
POS_SCORES = [0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6, 0.55, 0.5, 0.4]
NEG_SCORES = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6]

def cost_at(t, cfp, cfn, pos=POS_SCORES, neg=NEG_SCORES):
    """阈值 t：score >= t 判为正。返回期望代价 cfp*FP数 + cfn*FN数。"""
    fp = sum(1 for s in neg if s >= t)
    fn = sum(1 for s in pos if s < t)
    return cfp * fp + cfn * fn

# 三个阈值的代价对比（cfp=cfn=1 时，中间阈值通常最省）
for t in (0.3, 0.5, 0.7):
    print(f't={t:.1f}  cost(cfp=1,cfn=1)={cost_at(t, 1, 1)}   cost(cfp=1,cfn=10)={cost_at(t, 1, 10)}')

assert cost_at(0.5, 1, 1) == 3
assert cost_at(0.5, 1, 10) > cost_at(0.3, 1, 10)   # cfn 权重高时，阈值 0.5 比更低的阈值代价更大（漏检更贵）
print('\n✅ cost_at 就位：练习 2 会用它做网格搜索，找出最优阈值。')

## 4 · 延迟预算分解器

把一个端到端延迟预算，按权重分配到各子系统，并保证每个子系统至少拿到一个 `floor`（最低保证）。
**最后一个子系统吸收取整误差**，保证总和严格等于 `total`。

In [ ]:
def allocate_budget(total, weights, floors=None):
    """weights: {子系统: 相对权重}；floors: {子系统: 最低 ms}（可选）。
    先扣掉所有 floor，剩余按权重比例分配，最后一个子系统吸收取整误差。"""
    stages = list(weights.keys())
    floors = floors or {s: 0 for s in stages}
    floor_sum = sum(floors.get(s, 0) for s in stages)
    assert floor_sum <= total, 'floors 之和超过了总预算'
    remaining = total - floor_sum
    wsum = sum(weights.values())
    alloc, acc = {}, 0
    for s in stages[:-1]:
        v = floors.get(s, 0) + int(round(remaining * weights[s] / wsum))
        alloc[s] = v
        acc += v
    alloc[stages[-1]] = total - acc
    return alloc

# TSR 感知子系统：预处理/推理/后处理/跨模态融合，权重按经验耗时占比给出
weights = {'preprocess': 1, 'inference': 6, 'postprocess': 1, 'fusion': 2}
floors  = {'preprocess': 1, 'inference': 2, 'postprocess': 1, 'fusion': 1}
alloc = allocate_budget(33, weights, floors)

assert alloc == {'preprocess': 4, 'inference': 19, 'postprocess': 4, 'fusion': 6}
assert sum(alloc.values()) == 33

for s, v in alloc.items():
    print(f'  {s:<12} {v:>3} ms')
print(f'  {"合计":<12} {sum(alloc.values()):>3} ms   (预算 33ms)')
print('\n✅ 预算分解器就位：任何"再加一层"的提议，都能立刻被问"从哪个子系统的预算里扣"。')

## ✏️ 练习 1：一致性检查（指标背离检测器）

实现 `flag_divergence(metric_eval, metric_real, cost_eval, cost_real, metric_tol=0.01, cost_tol=0.10)`：

- `metric_delta = |metric_real - metric_eval|`
- `cost_rel = (cost_real - cost_eval) / cost_eval`
- 当 `metric_delta <= metric_tol` **且** `cost_rel > cost_tol` 时返回 `True`（模型指标看起来没变，业务代价却明显变差 —— 隐藏的背离）
- 否则返回 `False`

In [ ]:
def flag_divergence(metric_eval, metric_real, cost_eval, cost_real, metric_tol=0.01, cost_tol=0.10):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert flag_divergence(wr_eval, wr_real, bc_eval, bc_real) == True     # 本模块 §2 的真实数值：模型指标几乎不变，代价涨 25%+
assert flag_divergence(0.90, 0.92, 0.20, 0.18) == False                # 两者同向改善，没有背离
assert flag_divergence(0.90, 0.95, 0.20, 0.30) == False                # 模型指标本身也大幅变化，不算"隐藏的"背离

print('真实数值例子 ->', flag_divergence(wr_eval, wr_real, bc_eval, bc_real))
print('同向改善例子 ->', flag_divergence(0.90, 0.92, 0.20, 0.18))
print('\n✅ 练习 1 通过：只报一个模型指标永远不足以回答"业务变好了没有"。')

## ✏️ 练习 2：网格搜索最优工作点

实现 `optimal_threshold(cfp, cfn, pos=POS_SCORES, neg=NEG_SCORES)`：
在 `sorted(set(pos + neg))` 的候选阈值里，用 `cost_at` 找出**期望代价最小**的阈值，
返回 `(best_t, best_cost)`。若有并列最小值，取**候选顺序中先出现（更小）的那个**（即严格 `<` 才更新）。

In [ ]:
def optimal_threshold(cfp, cfn, pos=POS_SCORES, neg=NEG_SCORES):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
t_balanced, c_balanced = optimal_threshold(1, 1)
t_fn_heavy, c_fn_heavy = optimal_threshold(1, 10)    # 漏检代价远高于误检
t_fp_heavy, c_fp_heavy = optimal_threshold(10, 1)    # 误检代价远高于漏检

assert (t_balanced, c_balanced) == (0.5, 3)
assert (t_fn_heavy, c_fn_heavy) == (0.4, 4)
assert (t_fp_heavy, c_fp_heavy) == (0.65, 4)
assert t_fn_heavy < t_balanced < t_fp_heavy, '漏检代价越高，最优阈值越低（宁可多报警）'

print(f'cfp=cfn=1  (对称)         -> 阈值 {t_balanced}, 代价 {c_balanced}')
print(f'cfn=10x cfp (漏检更贵)    -> 阈值 {t_fn_heavy}, 代价 {c_fn_heavy}   <- 更低的阈值，抓得更多')
print(f'cfp=10x cfn (误检更贵)    -> 阈值 {t_fp_heavy}, 代价 {c_fp_heavy}   <- 更高的阈值，更谨慎')
print('\n✅ 练习 2 通过：这正是"漏检安全关键标志代价高，所以用更低阈值"这句话背后的计算。')

## ✏️ 练习 3：范围裁剪报告

实现 `scope_report(all_candidates, in_scope, reasons)`：

- `all_candidates`：候选功能点列表（全集）
- `in_scope`：本版本包含的功能点集合
- `reasons`：`{排除的功能点: 排除理由}` 字典（不一定覆盖所有排除项）

返回 `{'in_scope': [...], 'non_goals': [{'item':.., 'reason':..(可能是 None)}], 'missing_reason': [...]}`，
其中 `missing_reason` 是**被排除但没有给出理由**的功能点（这是加分项检查——"不做什么"必须说明为什么）。
三个列表都保持 `all_candidates` 中的原始顺序。

In [ ]:
def scope_report(all_candidates, in_scope, reasons):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ALL_CANDIDATES = ['static_signs', 'electronic_signs', 'multi_camera_fusion', 'hd_map_prior', 'rare_regional_signs']
IN_SCOPE = {'static_signs'}
REASONS = {
    'electronic_signs': '刷新率处理是独立子问题',
    'multi_camera_fusion': '标定与同步成本高',
    'hd_map_prior': '依赖地图团队接口',
    # 注意：'rare_regional_signs' 故意没给理由
}

report = scope_report(ALL_CANDIDATES, IN_SCOPE, REASONS)
assert report['in_scope'] == ['static_signs']
assert [ng['item'] for ng in report['non_goals']] == ['electronic_signs', 'multi_camera_fusion', 'hd_map_prior', 'rare_regional_signs']
assert report['missing_reason'] == ['rare_regional_signs']

print('本版本范围:', report['in_scope'])
print('明确不做（含理由）:')
for ng in report['non_goals']:
    print(f"  - {ng['item']:<22} {ng['reason'] or '⚠️ 缺理由'}")
print(f"\n缺理由的排除项: {report['missing_reason']}  <- 面试里被追问'为什么不做'会答不上来的地方")
print('\n✅ 练习 3 通过：范围裁剪的加分项不是"说了不做什么"，是"每一条都能说出为什么"。')

## ✏️ 练习 4：非功能需求覆盖度检查

实现 `nonfunctional_coverage(mentioned)`：给定一次设计演练里提到过的非功能需求维度集合 `mentioned`
（取值来自 `{'interpretability','compliance','privacy','rollback'}`），返回**没有被提到**的维度列表（按字母序排序）。

In [ ]:
def nonfunctional_coverage(mentioned):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert nonfunctional_coverage({'interpretability', 'rollback'}) == ['compliance', 'privacy']
assert nonfunctional_coverage(set()) == ['compliance', 'interpretability', 'privacy', 'rollback']
assert nonfunctional_coverage({'interpretability', 'compliance', 'privacy', 'rollback'}) == []

talked_about = {'interpretability', 'rollback'}   # TSR 场景里关联度最高的两项（见 §8）
missing_nf = nonfunctional_coverage(talked_about)
print(f'提到过: {sorted(talked_about)}')
print(f'还没提到: {missing_nf}')
print('\n✅ 练习 4 通过：§8 建议 TSR 场景优先讲可解释性与可回滚，但这份清单能告诉你还漏了什么。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def flag_divergence(metric_eval, metric_real, cost_eval, cost_real, metric_tol=0.01, cost_tol=0.10):
    metric_delta = abs(metric_real - metric_eval)
    cost_rel = (cost_real - cost_eval) / cost_eval
    return metric_delta <= metric_tol and cost_rel > cost_tol

In [ ]:
# 练习 2 参考答案
def optimal_threshold(cfp, cfn, pos=POS_SCORES, neg=NEG_SCORES):
    candidates = sorted(set(pos + neg))
    best_t, best_c = None, None
    for t in candidates:
        c = cost_at(t, cfp, cfn, pos, neg)
        if best_c is None or c < best_c:
            best_t, best_c = t, c
    return best_t, best_c

In [ ]:
# 练习 3 参考答案
def scope_report(all_candidates, in_scope, reasons):
    non_goals, missing_reason = [], []
    for item in all_candidates:
        if item in in_scope:
            continue
        reason = reasons.get(item)
        non_goals.append({'item': item, 'reason': reason})
        if reason is None:
            missing_reason.append(item)
    return {
        'in_scope': [c for c in all_candidates if c in in_scope],
        'non_goals': non_goals,
        'missing_reason': missing_reason,
    }

In [ ]:
# 练习 4 参考答案
def nonfunctional_coverage(mentioned):
    ALL_NF = {'interpretability', 'compliance', 'privacy', 'rollback'}
    return sorted(ALL_NF - set(mentioned))

---
## 🧪 真实工程胶囊：45 分钟里"需求澄清"环节的开场脚本 + 一页纸规格模板

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 拿到题目后的开场话术（中 / EN，对应 §2 七个维度）
# ══════════════════════════════════════════════════════════════════════
# CN: 「在我开始设计之前，我想先确认几件事：
#      这个系统的用户/下游是谁？在什么场景下运行？
#      我们怎么定义成功——具体到哪个指标、多少算达标？
#      失败会是什么样子，漏检和误检哪个代价更大？
#      有哪些硬约束——延迟、算力、数据规模？
#      以及同样重要的——第一版我们明确不做什么？」
# EN: "Before I start designing, let me confirm a few things: who is the
#      end user or downstream consumer? What's the operating context?
#      How do we define success — which metric, what threshold?
#      What do failures look like, and is a miss worse than a false alarm?
#      What are the hard constraints — latency, compute, data volume?
#      And just as important — what are we explicitly NOT doing in v1?"

# ══════════════════════════════════════════════════════════════════════
# B. 一页纸可验收规格模板（澄清完，当场写在白板角落）
# ══════════════════════════════════════════════════════════════════════
# 用户/下游      : ________________________
# 场景/约束      : 延迟 ____ms | 算力 ____ | 数据规模 ____
# 成功指标（三层）: 业务 ______ / 模型 ______ / 系统 ______
# 失败与代价      : 漏检代价 ____ : 误检代价 ____ (比值)
# 本版本范围      : 做 ______________
# 明确不做        : ______________ (理由: ______)
# 非功能需求      : 可解释 [ ] 合规 [ ] 隐私 [ ] 可回滚 [ ]

# ══════════════════════════════════════════════════════════════════════
# C. 常见追问与应对（对应正文 §4-§5）
# ══════════════════════════════════════════════════════════════════════
# Q: 「模型指标涨了，你怎么知道业务指标会跟着涨？」
# A: 「不一定跟着涨——常见的背离原因有评测集分布偏差、指标定义口径不同、
#      时间尺度不同、阈值不一致、多级复合误差。我们需要按真实流量分布和
#      业务代价重新加权算一遍，两者结论一致才能下结论。」
# Q: 「阈值怎么定？」
# A: 「取决于代价矩阵——漏检和误检哪个更贵。代价不对称时不会用统一阈值，
#      安全关键类别用更低阈值（宁可多报警），非关键类别用更高阈值控制误报。」

# ══════════════════════════════════════════════════════════════════════
# D. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 通用版"把模糊问题收敛成有边界的题"      -> C65 模块 04（本课不重复）
# · 代价不对称评测的完整方法（分桶/FP-per-km）-> C55 模块 05
# · 长尾类别的算法解法（重采样/重加权）      -> C58 模块 01
# · 数据系统设计（标注/泄漏/冷启动）        -> C63 模块 02（下一站）
# · 延迟预算的精确估算与容量规划            -> C63 模块 04
'''
print(RECIPE)
for token in ['一页纸可验收规格模板', '代价矩阵', 'C65 模块 04', 'C55 模块 05', 'C63 模块 02']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：开场话术 / 规格模板 / 追问应对 / 课程分工')

### 小结

- **需求澄清的产出不是一句话，是一份「可验收规格」**：一份写下来之后，双方都能判断"这个方案算不算做到了"的具体描述——
  七个维度（用户/场景/成功/失败/代价/约束/不做什么）缺一个都是隐患。
- **成功指标必须拆成三层**（业务指标 → 模型指标 → 系统指标），因为三者衡量的是不同的对象——
  模型指标是静态数据集上的统计量，系统指标叠加了级联与工程因素，业务指标才是最终结果。
  **只报一个模型指标，永远回答不了"业务真的变好了没有"这个问题。**
- **三层指标会背离，背后是五类结构性原因**（评测集分布偏差 / 定义口径不同 / 时间尺度不同 / 阈值不一致 / 复合误差）——
  本模块的数值例子证明了模型指标几乎不变（+0.16pp）时，业务代价可以上升 25%以上。
- **代价不对称直接决定工作点**：漏检代价越高于误检代价，最优阈值越低——这不是训练完之后的调参细节，
  而是需求澄清阶段就该问清楚、并在设计里显式体现的产出。
- **「不做什么」要具体、要给理由**：范围裁剪不是能力不够的表现，是让 45 分钟能讲完核心设计的必要动作；
  非功能需求（可解释/合规/隐私/可回滚）主动提到即是加分项，但不必四项平均用力，挑场景相关度最高的讲透即可。

下一站：**模块 02 · 数据系统设计** —— 从"指标定义好了"走到"拿什么数据去够到这个指标"：
数据来源与标注体系、划分与泄漏防范、长尾与冷启动策略。